# 第 1 周末练习 —— 技术概念解释器（Ollama）

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个软件工程 / 技术概念问题（例如「用类比解释 Transformer」）
- **输出**：清晰、易懂的解释（以 Markdown 展示）
- **后端**：本笔记本实际走的是 **本地 Ollama 的 OpenAI 兼容接口**（`base_url=http://localhost:11434/v1/`）

这是你在课程期间自己也能天天用的工具：遇到不懂的概念，改 `usr_prompt` 再跑一遍。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| OpenAI 兼容 `base_url` | 同一套 `OpenAI` SDK，指向本地 Ollama |
| 本地开源模型 | `llama3.2`（常量 `MODEL_LLAMA`） |
| 展示 Markdown | `display(Markdown(resp))` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 确保本机 Ollama 已启动，并已 `ollama pull llama3.2`
3. 在「提问」单元格改写 `usr_prompt`（或 `sys_prompt`），再跑调用格
4. 常量里也定义了 `MODEL_GPT`，但当前实现默认用的是 `MODEL_LLAMA`


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables）；本格先导入备用
import os
# 从 openai 导入 OpenAI 客户端类：既可调云端，也可通过 base_url 调本地 Ollama 兼容接口
from openai import OpenAI
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display（流式更新时用）
from IPython.display import Markdown, display, update_display
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（本练习主路径走本地时可不用密钥）
from dotenv import load_dotenv


In [2]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型名（本笔记本后续单元格未直接使用，留作对照 / 扩展）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [4]:
# ========== 创建客户端：指向本地 Ollama 的 OpenAI 兼容端点 ==========

# 用 OpenAI SDK，但 base_url 指到本机 11434 的 /v1/（不是 api.openai.com）
# api_key 对本地 Ollama 多为占位字符串；这里原代码写成 "llama3.2"（保持原样，不擅自改成 ollama）
openai=OpenAI(
  api_key="llama3.2",
  base_url="http://localhost:11434/v1/"
)


In [10]:
# ========== 封装一次提问：拼 messages → Chat Completions → 取文本 ==========

def ask_model(sys_prompt, usr_prompt):
  # 再次写出本地兼容端点地址（本函数内未直接使用；保留原变量，不删不改）
  model_url =  'http://localhost:11434/v1/'
  # messages：system 定角色/风格，user 放具体问题（Chat Completions 标准结构）
  msg = [{'role':'system', 'content':sys_prompt},{'role':'user', 'content':usr_prompt}]
  # 调用本地 llama3.2；返回的是完整 completion（非 stream）
  response = openai.chat.completions.create(model=MODEL_LLAMA, messages=msg)
  # 取出第一条 choice 的助手回复正文（字符串）
  return response.choices[0].message.content


In [13]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# system：规定助手身份与任务风格；发给模型的内容保持英文（改译会改变模型行为）
sys_prompt = "You are a helpful assistant who helps me understand software engineering concepts.\n"
# user：具体要解释的概念；可用类比（analogy）降低门槛
# 练习建议：换成你今天卡住的概念（如 attention / embedding），再跑下一格
usr_prompt = "Using a simple analogy, please explain the concept of Transformer architecture."


In [14]:
# ========== 调用模型并用 Markdown 展示回答 ==========

# 把上一格的 system / user 提示交给 ask_model，拿到完整回复字符串
resp = ask_model(sys_prompt, usr_prompt)
# 在 Jupyter 里把 Markdown 渲染出来（比纯 print 更易读）
display(Markdown(resp))


I'd be happy to help you understand the Transformer architecture using an analogy.

Imagine you're trying to translate a message from English to Spanish. You need to find the correct words in both languages that match each other, while also understanding the context and nuances of the conversation.

**Traditional Architecture: RNNs (Recurrent Neural Networks)**
In traditional NLP tasks like machine translation, we used to use RNNs. These are essentially "memory-based" models that rely on previous inputs to generate the next output. It's like using a notebook where you write down each word as you translate it, and then try to find the correct equivalent word in Spanish based on what you've written already.

For example, if we're translating the sentence "Hello, how are you?", the RNN model would look something like this:

... (write down "Hello" in English notebook)
... (write down "hello" in Spanish notebook) -> find similarity with previous output
... (write down "how to say that" in English notebook, maybe write down a word or phrase)
... (write down the translation of the phrase in the Spanish notebook)

**Transformer Architecture**
Now, imagine using a completely new approach. Instead of relying on previous inputs, we focus on the entire sequence of words at once and use self-attention mechanisms to find relationships between them.

In the Transformer architecture, we focus on three key aspects for each word in the input sentence:

1. **Self-attention**: We look at all other words simultaneously to see how similar they are to our current word in terms of meaning.
2. **Query**: Each word acts as a "query" pointing to its relevant context.
3. **Score**: A weighted sum that captures strengths and weaknesses between words.

We multiply these together (think of it like a matrix product) to generate an output representation that combines all the contextual information from each other word. In our English-to-Spanish translation example, this would look something like:

... "Hello" -> self-attention relationships with surrounding words
... calculate query vectors combining individual context tokens
... multiply and aggregate results (weighted scores)

The key insights from Transformer are:

* **Parallel processing**: We process the entire input sequence simultaneously, which leads to significant speedup in training times.
* **Self-attention mechanism**: This innovative attention layer efficiently captures long-range relationships between words by reducing the need for recurrent neural networks' sequential dependencies.

This analogy is certainly oversimplified, but it should give you an idea of how Transformers differ from traditional RNN-based architectures.